In [1]:
import numpy as np

# set global print options
np.set_printoptions(suppress=True, precision=3)

## 1. Rank of Summed and Multiplied Matrices
Based on [Lecture 64](https://wgu.udemy.com/course/linear-algebra-theory-and-implementation/learn/lecture/10500648#overview) and [Lecture 68](https://wgu.udemy.com/course/linear-algebra-theory-and-implementation/learn/lecture/10500648#overview), these rules define the maximum possible information (dimensions) preserved through operations.

* **Multiplication Rule (The Bottleneck):**
    $$rank(AB) \leq \min(rank(A), rank(B))$$
    * *Logic:* You cannot create more information than what is available in the "narrowest" part of the data pipeline.

* **Addition Rule:**
    $$rank(A + B) \leq rank(A) + rank(B)$$
    * *Logic:* Summing two matrices can combine their spans, but the total rank can never exceed the sum of the individual ranks.

In [35]:
LEFT_MATRIX_SHAPE = (7, 4)
RIGHT_MATRIX_SHAPE = (4, 7)


def create_rank_deficient_matrix(shape):
    """Create a random matrix with at least one dependent row."""
    # Start with a random matrix of the requested shape.
    matrix = np.random.rand(*shape)

    # Copy the second-to-last row into the last row.
    # This makes one row linearly dependent, so the matrix cannot have full row rank.
    matrix[-1, :] = matrix[-2, :]

    return matrix


def print_matrix_with_rank(label, matrix):
    """Print a matrix and its numerical rank."""
    # NumPy computes numerical rank using singular values.
    matrix_rank = np.linalg.matrix_rank(matrix)

    print(f"{label}:")
    print(matrix)
    print(f"Rank of {label.lower()}: {matrix_rank}\n")


# Create a 7x4 matrix.
# A random rectangular matrix usually has full possible rank: min(7, 4) = 4.
left_matrix = np.random.rand(*LEFT_MATRIX_SHAPE)

# Create a 4x7 matrix with a repeated row.
# Because one row is duplicated, its rank should be less than the maximum possible rank of 4.
right_matrix = create_rank_deficient_matrix(RIGHT_MATRIX_SHAPE)

# Add matrices with matching dimensions.
# right_matrix is 4x7, so transposing it gives 7x4, matching left_matrix.
summed_matrix = left_matrix + right_matrix.T

# Matrix multiplication is valid because the inner dimensions match:
# left_matrix is 7x4 and right_matrix is 4x7, so the product is 7x7.
product_matrix = left_matrix @ right_matrix

print("Original matrices:\n")
print_matrix_with_rank("Left matrix", left_matrix)
print_matrix_with_rank("Right matrix", right_matrix)

# The rank of a sum is bounded by the sum of the individual ranks,
# but it also cannot exceed the matrix's maximum possible rank.
print_matrix_with_rank("Summed matrix", summed_matrix)

# The rank of a product is bounded by the lower-rank factor.
# Since right_matrix is intentionally rank-deficient, the product rank should reflect that bottleneck.
print_matrix_with_rank("Product matrix", product_matrix)


Original matrices:

Left matrix:
[[0.656 0.296 0.076 0.805]
 [0.32  0.888 0.355 0.095]
 [0.533 0.276 0.211 0.093]
 [0.948 0.779 0.633 0.578]
 [0.248 0.896 0.273 0.067]
 [0.253 0.748 0.606 0.607]
 [0.646 0.257 0.573 0.006]]
Rank of left matrix: 4

Right matrix:
[[0.478 0.989 0.751 0.276 0.229 0.066 0.456]
 [0.039 0.127 0.121 0.039 0.539 0.463 0.47 ]
 [0.497 0.462 0.939 0.474 0.372 0.193 0.667]
 [0.497 0.462 0.939 0.474 0.372 0.193 0.667]]
Rank of right matrix: 3

Summed matrix:
[[1.134 0.335 0.573 1.302]
 [1.309 1.015 0.817 0.557]
 [1.283 0.397 1.151 1.033]
 [1.225 0.818 1.107 1.052]
 [0.477 1.435 0.645 0.438]
 [0.319 1.211 0.799 0.8  ]
 [1.103 0.726 1.239 0.672]]
Rank of summed matrix: 4

Product matrix:
[[0.763 1.094 1.356 0.61  0.637 0.35  1.026]
 [0.411 0.637 0.77  0.336 0.718 0.519 0.863]
 [0.417 0.703 0.719 0.302 0.384 0.222 0.576]
 [1.085 1.597 1.944 0.866 1.087 0.656 1.606]
 [0.322 0.516 0.614 0.264 0.666 0.497 0.761]
 [0.753 0.906 1.42  0.674 0.912 0.597 1.276]
 [0.606 0.939 1.

In [ ]:
MATRIX_SIZE = 5

# Use NumPy's newer random number generator API.
# Keeping it in one variable makes it easy to reuse throughout the notebook.
rng = np.random.default_rng()


def generate_symmetric_matrix(size, random_generator):
    """Generate a square symmetric matrix with random values."""
    # Start with a random square matrix.
    random_matrix = random_generator.random((size, size))

    # Add the matrix to its transpose so entry (i, j) equals entry (j, i).
    return random_matrix + random_matrix.T


def matrix_rank(matrix):
    """Return the numerical rank of a matrix."""
    # NumPy estimates rank numerically using singular values.
    # This is more reliable than trying to row-reduce floating-point matrices by hand.
    return np.linalg.matrix_rank(matrix)


def gram_matrix(matrix):
    """Return matrix @ matrix.T."""
    # A @ A.T is a Gram-style matrix.
    # It preserves rank relationships that are useful for testing:
    # rank(A) = rank(A.T) = rank(A.T @ A) = rank(A @ A.T)
    return matrix @ matrix.T


def format_rank(label, matrix):
    """Format a matrix rank line for display."""
    return f"Rank of {label}: {matrix_rank(matrix)}"


def print_matrix_rank_summary(matrix, matrix_name="A"):
    """Print rank relationships for a matrix, its transpose, and Gram products."""
    # The transpose should have the same rank as the original matrix.
    transpose_matrix = matrix.T

    # These products are square matrices built from A.
    # For a given matrix A, A.T @ A and A @ A.T have the same rank as A.
    transpose_product = transpose_matrix @ matrix
    product_with_transpose = gram_matrix(matrix)

    print(format_rank(matrix_name, matrix))
    print(format_rank(f"{matrix_name}^T{matrix_name}", transpose_product))
    print(format_rank(f"{matrix_name}{matrix_name}^T", product_with_transpose))
    print(format_rank(f"{matrix_name}^T", transpose_matrix))


# Create two symmetric matrices to compare rank behavior under addition and multiplication.
first_matrix = generate_symmetric_matrix(MATRIX_SIZE, rng)
second_matrix = generate_symmetric_matrix(MATRIX_SIZE, rng)

# First, verify the standard rank relationships for A, A.T, A.T @ A, and A @ A.T.
print_matrix_rank_summary(first_matrix)

# Build Gram-style matrices from both symmetric matrices.
first_gram_matrix = gram_matrix(first_matrix)
second_gram_matrix = gram_matrix(second_matrix)

# Addition can combine column/row spaces, but rank is still bounded.
gram_matrix_sum = first_gram_matrix + second_gram_matrix

# Multiplication cannot exceed the rank of the lowest-rank factor.
gram_matrix_product = first_gram_matrix @ second_gram_matrix

print(format_rank("the sum of AA^T + BB^T", gram_matrix_sum))
print(format_rank("the product of AA^T and BB^T", gram_matrix_product))